In [ ]:
# Pretraining LLMs

In [ ]:
# 打印本章依赖库的版本，确保环境一致（matplotlib 画图 / numpy 数值 / tiktoken 分词 / torch 深度学习框架）
from importlib.metadata import version

pkgs = ["matplotlib",
        "numpy",
        "tiktoken",
        "torch",
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
#Using GPT to generate text

In [ ]:
# 导入 PyTorch 与前几章封装好的 GPT 模型类（定义在 supplementary.py）
import torch
from supplementary import GPTModel

In [ ]:
# 定义 124M 参数规模的 GPT 配置，并实例化模型。
# 注意：为节省算力，context_length 缩短为 256（原始 GPT-2 为 1024）。
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size          # 词表大小（GPT-2 BPE）
    "context_length": 256, # Shortened context length (orig: 1024)  # 最大上下文长度（缩短版）
    "emb_dim": 768,        # Embedding dimension       # 词/隐藏向量维度
    "n_heads": 12,         # Number of attention heads # 注意力头数
    "n_layers": 12,        # Number of layers          # Transformer 层数
    "drop_rate": 0.1,      # Dropout rate              # Dropout 比例
    "qkv_bias": False      # Query-key-value bias      # Q/K/V 线性层是否带偏置
}

torch.manual_seed(123)              # 固定随机种子，保证权重初始化可复现
model = GPTModel(GPT_CONFIG_124M)   # 按配置构建随机初始化的模型
model.eval();  # Disable dropout during inference   # 评估模式：关闭 Dropout，用于确定性推理

In [ ]:
# 文本 <-> token id 转换的辅助函数（此处在 notebook 内重新定义，与 supplementary.py 中同名）。
# 关键在于 batch 维度的增删：模型 forward 需要 (batch, seq_len) 形状。
import tiktoken
from supplementary import generate_text_simple
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})  # 文本 -> id 列表 (seq_len,)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension  # -> (1, seq_len)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension  # (1, seq_len) -> (seq_len,)
    return tokenizer.decode(flat.tolist())  # id -> 文本

In [ ]:
# 用「未训练」的随机模型生成文本：此时输出必然是乱码，作为训练前的基线对照。
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")   # GPT-2 分词器

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),   # 起始文本 -> (1, seq_len)
    max_new_tokens=10,                                  # 续写 10 个 token
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
#Preparing the dataset loaders

In [ ]:
# 读取训练语料：短篇小说《The Verdict》，作为本章的小规模预训练数据集。
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    text_data = file.read()

In [ ]:
# First 100 characters
# 查看文本开头，确认读取正确
print(text_data[:99])

In [ ]:
# Last 100 characters
# 查看文本结尾
print(text_data[-99:])

In [ ]:
# 统计字符数与 token 数。BPE 分词后 token 数通常少于字符数（一个 token 常对应多个字符）。
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("Characters:", total_characters)
print("Tokens:", total_tokens)

In [ ]:
# 构造训练/验证 DataLoader。
# 关键点1：按 90/10 比例切分文本为训练集与验证集。
# 关键点2：stride == context_length，使各样本窗口首尾相接、互不重叠，最大化利用数据且避免重复。
from supplementary import create_dataloader_v1


# Train/validation ratio
train_ratio = 0.90                              # 90% 训练，10% 验证
split_idx = int(train_ratio * len(text_data))   # 切分位置（按字符）
train_data = text_data[:split_idx]              # 前 90%
val_data = text_data[split_idx:]                # 后 10%


torch.manual_seed(123)   # 固定种子，使训练集的 shuffle 可复现

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],   # stride=上下文长度 -> 窗口不重叠
    drop_last=True,                             # 丢弃不满一批的尾部，保证 batch 大小一致
    shuffle=True,                               # 训练集打乱
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,                            # 验证集保留全部数据
    shuffle=False,                              # 验证集不打乱
    num_workers=0
)

In [ ]:
# 打印各批次的输入/目标张量形状，验证 DataLoader。
# 每个批次形状均为 (batch_size, context_length) = (2, 256)；输入 x 与目标 y 形状相同（y 是 x 右移一位）。
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

In [ ]:
# 统计训练/验证集实际参与的 token 总数（numel() = 张量元素个数 = batch_size * seq_len 累加）。
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

In [ ]:
# 计算「训练前」的初始损失作为基线。
# 原理：随机初始化的模型对每个 token 的预测接近均匀分布，交叉熵约为 ln(vocab_size)=ln(50257)≈10.8。
# 训练的目标就是把这个损失显著降下来。
from supplementary import calc_loss_loader


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # 优先用 GPU
model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes  # nn.Module 原地搬移


torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader  # 复现 shuffle

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet  # 仅评估，不建计算图
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

In [ ]:
#Training an LLM

In [ ]:
# 【本章核心】定义训练主循环 train_model_simple。
# 标准训练流程：对每个 epoch、每个 batch 执行
#   zero_grad -> 前向算 loss -> loss.backward() 反向传播 -> optimizer.step() 更新参数。
# 并周期性（每 eval_freq 步）评估训练/验证损失并记录，用于后续画损失曲线。
#
# ✅ 已修复 Bug 1：`return ...` 原在 `for epoch` 循环体内（8 空格）——第 1 轮后就返回，num_epochs 只跑 1 轮；
#    现移到两层 for 循环之外、函数末尾（4 空格），使 num_epochs 轮全部执行。
# ✅ 已修复 Bug 2：`generate_and_print_sample(...)` 原在内层 batch 循环内（12 空格，每 batch 生成一次）；
#    现移到 batch 循环之外、epoch 循环之内（8 空格），改为每个 epoch 结束时生成一次样本。
from supplementary import (
    calc_loss_batch,
    evaluate_model,
    generate_and_print_sample
)
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    train_losses,val_losses,train_tokens_seen=[],[],[]   # 记录损失与已见 token 数，供绘图
    tokens_seen, global_step = 0, -1                      # 累计 token 数、全局步数（从 -1 起，首步变 0）
    for epoch in range(num_epochs):
        model.train()                                    # 每个 epoch 开始切到训练模式（开启 Dropout）
        for input_batch,target_batch in train_loader:
            optimizer.zero_grad()                        # 清空上一步的梯度（否则会累加）
            loss=calc_loss_batch(input_batch,target_batch,model,device)  # 前向 + 交叉熵损失
            loss.backward()                              # 反向传播计算梯度
            optimizer.step()                             # 按梯度更新参数
            tokens_seen+=input_batch.numel()             # 累加本批次处理的 token 数
            global_step+=1
            if global_step% eval_freq==0:                # 每 eval_freq 步评估一次
                train_loss,val_loss=evaluate_model(model,train_loader,val_loader,device,eval_iter)
                train_losses.append(train_loss)
                train_tokens_seen.append(tokens_seen)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")
        # 每个 epoch 结束时（已移出 batch 循环）生成一次样本，直观观察训练进展。
        generate_and_print_sample(model,tokenizer,device,start_context)
    # 函数末尾（两层循环之外）才返回，确保 num_epochs 轮全部跑完。
    return train_losses,val_losses,train_tokens_seen

In [ ]:
# 【本章核心】实例化模型与优化器并启动训练。
# AdamW：Adam 的改进版，将 weight_decay（权重衰减，即 L2 正则）与梯度更新解耦，是 GPT 训练的标准选择。
# weight_decay=0.1 施加正则以抑制过拟合；lr=0.0004 为学习率。
# 注意：由于上面单元的 Bug 1，num_epochs=10 实际只会训练 1 个 epoch。
torch.manual_seed(123)
model=GPTModel(GPT_CONFIG_124M)                       # 重新初始化一个全新模型用于训练
model.to(device)
optimizer=torch.optim.AdamW(model.parameters(),lr=0.0004,weight_decay=0.1)  # AdamW 优化器
num_epochs=10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,   # 每 5 步评估，评估时只用 5 个 batch 估算
    start_context="Every effort moves you", tokenizer=tokenizer
)

In [ ]:
# 保存模型权重。state_dict() 只保存参数张量（而非整个模型对象），是 PyTorch 推荐的持久化方式。
torch.save(model.state_dict(), "model.pth")

In [ ]:
# 绘制训练/验证损失曲线。
# epochs_tensor：把 [0, num_epochs] 均匀切成 len(train_losses) 个点，作为下轴（epoch）坐标；
# tokens_seen：作为上轴坐标。由于过拟合小数据集，通常会看到训练损失持续下降而验证损失回升。
from supplementary import plot_losses
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

In [ ]:
#Exercise 3 (Optional): Train the LLM on your own favorite texts

In [ ]:
# 用「训练后」的模型再次续写同一起始文本，与训练前的乱码对比，观察模型学到的语言能力。
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer).to(device),   # 起始文本 -> (1, seq_len) 并搬到 device
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

In [ ]:
# 解码并打印生成结果
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
# 演示如何从磁盘重新加载已保存的模型权重：
# 先按相同配置构建空模型，再用 load_state_dict 把参数灌入；map_location 确保权重加载到正确设备。
import torch

# Imports from a local file
from supplementary import GPTModel


model = GPTModel(GPT_CONFIG_124M)                                        # 结构必须与保存时一致
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("model.pth", map_location=device))     # 加载权重
model.eval();                                                          # 切到评估模式用于推理